# fexp_panel → Parquet

Read Barra GEM4US factor exposure panel from SAS v9, keep US / USMC names (`aci_us=1` or `aci_usmc=1`), write filtered rows to `./parquet_files/`.

**Source:** `Y:\sasdata\barra\gem4us\fexp_panel.sas7bdat`  
**Output:** `./parquet_files/fexp_panel.parquet`

SAS does not support predicate pushdown. We read in row windows with `pyreadstat` (`output_format="polars"`) and filter each chunk before accumulating — no pandas involved.

In [4]:
from pathlib import Path

import polars as pl
import pyreadstat

SAS_PATH = Path(r"Y:\sasdata\barra\gem4us\fexp_panel.sas7bdat")
OUTPUT_DIR = Path("parquet_files")
OUTPUT_PATH = OUTPUT_DIR / "fexp_panel.parquet"
CHUNK_SIZE = 500_000

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
def load_filtered_fexp_panel(
    sas_path: Path,
    chunk_size: int = CHUNK_SIZE,
) -> pl.DataFrame:
    """Scan SAS file in chunks; keep rows where aci_us=1 or aci_usmc=1."""
    filtered_chunks: list[pl.DataFrame] = []
    rows_read = 0
    rows_kept = 0

    sas_path_str = str(sas_path)
    _, meta = pyreadstat.read_sas7bdat(
        sas_path_str,
        metadataonly=True,
        output_format="polars",
    )
    num_rows = meta.number_rows

    for offset in range(0, num_rows, chunk_size):
        chunk, _meta = pyreadstat.read_sas7bdat(
            sas_path_str,
            row_offset=offset,
            row_limit=chunk_size,
            output_format="polars",
        )
        if chunk.height == 0:
            break

        rows_read += chunk.height
        kept = chunk.filter((pl.col("aci_us") == 1) | (pl.col("aci_usmc") == 1))
        if kept.height:
            filtered_chunks.append(kept)
            rows_kept += kept.height

    if not filtered_chunks:
        return pl.DataFrame()

    result = pl.concat(filtered_chunks, how="vertical_relaxed")
    print(f"Read {rows_read:,} rows; kept {rows_kept:,} ({rows_kept / rows_read:.1%})")
    return result

In [6]:
if not SAS_PATH.exists():
    raise FileNotFoundError(f"SAS dataset not found: {SAS_PATH}")

fexp_panel = load_filtered_fexp_panel(SAS_PATH)
fexp_panel.write_parquet(OUTPUT_PATH)

print(f"Wrote {fexp_panel.height:,} rows to {OUTPUT_PATH.resolve()}")
fexp_panel.head()

KeyboardInterrupt: 

In [ ]:
# Round-trip check
pl.scan_parquet(OUTPUT_PATH).select(pl.len()).collect()

len
u32
1238718
